## Bank Loan Approval Project using (Logistic Regression)

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
import warnings
warnings.filterwarnings('ignore')

In [3]:
loan = pd.read_csv('loan_data.csv')

In [4]:
loan

,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,female,Master,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,3.0,561,No,1
1,21.0,female,High School,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504,Yes,0
2,25.0,female,High School,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635,No,1
3,23.0,female,Bachelor,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675,No,1
4,24.0,male,Master,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586,No,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44995,27.0,male,Associate,47971.0,6,RENT,15000.0,MEDICAL,15.66,0.31,3.0,645,No,1
44996,37.0,female,Associate,65800.0,17,RENT,9000.0,HOMEIMPROVEMENT,14.07,0.14,11.0,621,No,1
44997,33.0,male,Associate,56942.0,7,RENT,2771.0,DEBTCONSOLIDATION,10.02,0.05,10.0,668,No,1
44998,29.0,male,Bachelor,33164.0,4,RENT,12000.0,EDUCATION,13.23,0.36,6.0,604,No,1


## Data Preprocessing

In [5]:
loan.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45000 entries, 0 to 44999
Data columns (total 14 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   person_age                      45000 non-null  float64
 1   person_gender                   45000 non-null  object 
 2   person_education                45000 non-null  object 
 3   person_income                   45000 non-null  float64
 4   person_emp_exp                  45000 non-null  int64  
 5   person_home_ownership           45000 non-null  object 
 6   loan_amnt                       45000 non-null  float64
 7   loan_intent                     45000 non-null  object 
 8   loan_int_rate                   45000 non-null  float64
 9   loan_percent_income             45000 non-null  float64
 10  cb_person_cred_hist_length      45000 non-null  float64
 11  credit_score                    45000 non-null  int64  
 12  previous_loan_defaults_on_file  

In [6]:
loan.corr(numeric_only=True)

,person_age,person_income,person_emp_exp,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,loan_status
person_age,1.000000,0.193698,0.954412,0.050750,0.013402,-0.043299,0.861985,0.178432,-0.021476
person_income,0.193698,1.000000,0.185987,0.242290,0.001510,-0.234177,0.124316,0.035919,-0.135808
person_emp_exp,0.954412,0.185987,1.000000,0.044589,0.016631,-0.039862,0.824272,0.186196,-0.020481
loan_amnt,0.050750,0.242290,0.044589,1.000000,0.146093,0.593011,0.042969,0.009074,0.107714
loan_int_rate,0.013402,0.001510,0.016631,0.146093,1.000000,0.125209,0.018008,0.011498,0.332005
loan_percent_income,-0.043299,-0.234177,-0.039862,0.593011,0.125209,1.000000,-0.031868,-0.011483,0.384880
cb_person_cred_hist_length,0.861985,0.124316,0.824272,0.042969,0.018008,-0.031868,1.000000,0.155204,-0.014851
credit_score,0.178432,0.035919,0.186196,0.009074,0.011498,-0.011483,0.155204,1.000000,-0.007647
loan_status,-0.021476,-0.135808,-0.020481,0.107714,0.332005,0.384880,-0.014851,-0.007647,1.000000


In [7]:
loan.columns

Index(['person_age', 'person_gender', 'person_education', 'person_income',
       'person_emp_exp', 'person_home_ownership', 'loan_amnt', 'loan_intent',
       'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length',
       'credit_score', 'previous_loan_defaults_on_file', 'loan_status'],
      dtype='object')

In [8]:
loan.drop(['person_education', 'person_emp_exp','person_home_ownership', 'cb_person_cred_hist_length']
                       ,axis=1, inplace=True)

In [9]:
loan.head()

,person_age,person_gender,person_income,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,female,71948.0,35000.0,PERSONAL,16.02,0.49,561,No,1
1,21.0,female,12282.0,1000.0,EDUCATION,11.14,0.08,504,Yes,0
2,25.0,female,12438.0,5500.0,MEDICAL,12.87,0.44,635,No,1
3,23.0,female,79753.0,35000.0,MEDICAL,15.23,0.44,675,No,1
4,24.0,male,66135.0,35000.0,MEDICAL,14.27,0.53,586,No,1


In [10]:
loan['loan_intent'].value_counts()

loan_intent
EDUCATION            9153
MEDICAL              8548
VENTURE              7819
PERSONAL             7552
DEBTCONSOLIDATION    7145
HOMEIMPROVEMENT      4783
Name: count, dtype: int64

In [11]:
loan['previous_loan_defaults_on_file'].value_counts()

previous_loan_defaults_on_file
Yes    22858
No     22142
Name: count, dtype: int64

In [12]:
loan['person_gender'].value_counts()

person_gender
male      24841
female    20159
Name: count, dtype: int64

In [13]:
loan['person_income']

0        71948.0
1        12282.0
2        12438.0
3        79753.0
4        66135.0
          ...   
44995    47971.0
44996    65800.0
44997    56942.0
44998    33164.0
44999    51609.0
Name: person_income, Length: 45000, dtype: float64

In [14]:
loan.columns

Index(['person_age', 'person_gender', 'person_income', 'loan_amnt',
       'loan_intent', 'loan_int_rate', 'loan_percent_income', 'credit_score',
       'previous_loan_defaults_on_file', 'loan_status'],
      dtype='object')

In [15]:
loan.head()

,person_age,person_gender,person_income,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,female,71948.0,35000.0,PERSONAL,16.02,0.49,561,No,1
1,21.0,female,12282.0,1000.0,EDUCATION,11.14,0.08,504,Yes,0
2,25.0,female,12438.0,5500.0,MEDICAL,12.87,0.44,635,No,1
3,23.0,female,79753.0,35000.0,MEDICAL,15.23,0.44,675,No,1
4,24.0,male,66135.0,35000.0,MEDICAL,14.27,0.53,586,No,1


## Feature  Engineering and Selection 

In [16]:
loan['person_gender'] = loan['person_gender'].map({'female': 0, 'male': 1}).astype(int)

In [17]:
loan['previous_loan_defaults_on_file'] = loan['previous_loan_defaults_on_file'].map({'No': 0, 'Yes': 1}).astype(int)

In [18]:
x = loan.drop('loan_status', axis=1)
y = loan['loan_status']

## Create column-Transformer and Pipeline

In [19]:
num_features = ['person_age','person_gender', 'person_income', 'loan_amnt', 'loan_int_rate', 
                'loan_percent_income','credit_score', 'previous_loan_defaults_on_file']
cat_feature = ['loan_intent']

In [20]:
preprocess = ColumnTransformer(
transformers=[('num', StandardScaler(), num_features),
                ('cat', OneHotEncoder(handle_unknown='ignore'), cat_feature)]
)

In [21]:
pipline = Pipeline([('preprcessing', preprocess),
                ('model', LogisticRegression())])

In [22]:
x_train, x_test, y_train, y_test = train_test_split(x,y, test_size=0.2, random_state=42)

In [23]:
x_train.head(1)

,person_age,person_gender,person_income,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,credit_score,previous_loan_defaults_on_file
25180,34.0,0,97265.0,15000.0,PERSONAL,12.73,0.15,631,0


In [24]:
pipline.fit(x_train, y_train)

,steps,"[('preprcessing', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [25]:
y_pred = pipline.predict(x_test)

In [26]:
y_pred

array([0, 0, 1, ..., 0, 1, 1], shape=(9000,))

In [27]:
from sklearn.metrics import accuracy_score

In [28]:
print('Accuracy', accuracy_score(y_test, y_pred))

Accuracy 0.8893333333333333


In [29]:
x_test.head(3)

,person_age,person_gender,person_income,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,credit_score,previous_loan_defaults_on_file
37979,32.0,1,96865.0,7500.0,EDUCATION,6.04,0.08,601,0
9911,24.0,1,56838.0,9000.0,EDUCATION,11.49,0.16,647,1
43386,22.0,0,37298.0,5000.0,MEDICAL,14.88,0.13,711,0


### New Dataset to predict our model

In [30]:
new_customer = pd.DataFrame({
    'person_age':[22.0],
    'person_gender' : [0], # means Female
    'person_income' : [37000],
    'loan_amnt' : [5000.0],
    'loan_intent' : ['MEDICAL'],
    'loan_int_rate' : [14.88],
    'loan_percent_income' : [0.13],
    'credit_score': [711],
    'previous_loan_defaults_on_file': [0] # NO
    
})

In [39]:
print('Prediction New Customer', pipline.predict(new_customer))

Prediction New Customer [1]


In [32]:
y_test.head()

37979    0
9911     0
43386    1
13822    0
44810    1
Name: loan_status, dtype: int64

### Confusion Matrix

In [33]:
from sklearn.metrics import confusion_matrix

In [34]:
confusion_matrix(y_test, y_pred)

array([[6551,  439],
       [ 557, 1453]])

## Evaluates models

In [35]:
from sklearn.metrics import classification_report

In [36]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.92      0.94      0.93      6990
           1       0.77      0.72      0.74      2010

    accuracy                           0.89      9000
   macro avg       0.84      0.83      0.84      9000
weighted avg       0.89      0.89      0.89      9000



## Threshold

In [37]:
y_prob = pipline.predict_proba(x_test)[:,1]
y_pred_06 = (y_prob > 0.6).astype(int)
confusion_matrix(y_test, y_pred_06)
print(classification_report(y_test, y_pred_06))

              precision    recall  f1-score   support

           0       0.90      0.96      0.93      6990
           1       0.81      0.63      0.71      2010

    accuracy                           0.88      9000
   macro avg       0.85      0.79      0.82      9000
weighted avg       0.88      0.88      0.88      9000



## Other Algorithm apply for checking better performance

In [96]:
from sklearn.ensemble import RandomForestClassifier

rf_pipeline = Pipeline(steps=[
    ('preprocessing', preprocess),
    ('model', RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight='balanced'
    ))
])

rf_pipeline.fit(x_train, y_train)

y_pred_rf = rf_pipeline.predict(x_test)

from sklearn.metrics import accuracy_score, classification_report
print("RF Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

RF Accuracy: 0.9191111111111111
              precision    recall  f1-score   support

           0       0.94      0.96      0.95      6990
           1       0.85      0.78      0.81      2010

    accuracy                           0.92      9000
   macro avg       0.89      0.87      0.88      9000
weighted avg       0.92      0.92      0.92      9000



In [97]:
from sklearn.ensemble import GradientBoostingClassifier

gb_pipeline = Pipeline(steps=[
    ('preprocessing', preprocess),
    ('model', GradientBoostingClassifier(random_state=42))
])

gb_pipeline.fit(x_train, y_train)

y_pred_gb = gb_pipeline.predict(x_test)

print("GB Accuracy:", accuracy_score(y_test, y_pred_gb))
print(classification_report(y_test, y_pred_gb))

GB Accuracy: 0.9138888888888889
              precision    recall  f1-score   support

           0       0.94      0.96      0.95      6990
           1       0.83      0.77      0.80      2010

    accuracy                           0.91      9000
   macro avg       0.88      0.86      0.87      9000
weighted avg       0.91      0.91      0.91      9000

